In [1]:
import cv2
import numpy as np
from ultralytics import YOLO

# Load the YOLOv8 model
model = YOLO('F:\insighteye\datacenter\yolov8l.pt')  # Replace with 'yolov8m.pt' for medium model


# Set the video source (0 for webcam, or provide a video file path)
video_source = 'F:/insighteye/datacenter/helmet/trafficTest.mp4'  # Update to video path if needed
cap = cv2.VideoCapture(video_source)

# Define the output video writer (optional)
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
out = cv2.VideoWriter('F:/insighteye/datacenter/helmet/trafficTestOutPut2.mp4', cv2.VideoWriter_fourcc(*'XVID'), 30, (frame_width, frame_height))

# Define colors for classes
colors = {
    'person': (0, 255, 0),  # Green for person
    'motorcycle': (255, 0, 0)   # Blue for motorcycle
}

# Class IDs for COCO dataset
person_class_id = 0  # Person class id 
motorcycle_class_id = 3  #motorcycle class id 

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Perform inference on the frame
    results = model(frame)

    person_count = 0
    motorcycle_count = 0

    for result in results:
        for box in result.boxes.data:
            x1, y1, x2, y2 = map(int, box[:4])  # Bounding box coordinates
            conf = float(box[4])                # Confidence score (as float)
            cls = int(box[5])                   # Class ID (as integer)

            # Check for 'person' class
            if cls == person_class_id:
                person_count += 1
                label = f'Person {conf:.2f}'  # Display confidence up to 2 decimals
                color = colors['person']
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            # Check for 'motorcycle' classes
            elif cls == motorcycle_class_id:
                motorcycle_count += 1
                label = f'Motorcycle {conf:.2f}'  # Display confidence up to 2 decimals
                color = colors['motorcycle']
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Display the counts on the frame
    cv2.putText(frame, f'Persons Detected: {person_count}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    cv2.putText(frame, f'motorcycle Detected: {motorcycle_count}', (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

    # Show the video frame
    cv2.imshow('Person and motorcycle detected', frame)

    # Write the frame to output video (optional)
    out.write(frame)

    # Break the loop on 'q' key press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
out.release()
cv2.destroyAllWindows()

<>:6: SyntaxWarning: invalid escape sequence '\i'
<>:6: SyntaxWarning: invalid escape sequence '\i'
C:\Users\Bobby\AppData\Local\Temp\ipykernel_16068\4220307086.py:6: SyntaxWarning: invalid escape sequence '\i'
  model = YOLO('F:\insighteye\datacenter\yolov8l.pt')  # Replace with 'yolov8m.pt' for medium model



0: 384x640 3 persons, 13 cars, 2 motorcycles, 1 bus, 2 trucks, 900.2ms
Speed: 5.0ms preprocess, 900.2ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 13 cars, 2 motorcycles, 2 trucks, 795.0ms
Speed: 3.0ms preprocess, 795.0ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 13 cars, 1 motorcycle, 1 truck, 787.5ms
Speed: 2.0ms preprocess, 787.5ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 13 cars, 1 motorcycle, 2 trucks, 757.6ms
Speed: 5.0ms preprocess, 757.6ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 13 cars, 1 motorcycle, 2 trucks, 723.4ms
Speed: 2.0ms preprocess, 723.4ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 14 cars, 1 motorcycle, 2 trucks, 740.7ms
Speed: 2.0ms preprocess, 740.7ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 person

In [2]:
# people + motorcyle + helmet
import cv2
import numpy as np
from ultralytics import YOLO

# Load the YOLOv8 model
model = YOLO('F:/insighteye/datacenter/helmet/runs/best.pt')  # Model with helmet detection capability

# Set the video source (0 for webcam, or provide a video file path)
video_source = 'F:/insighteye/datacenter/helmet/trafficTest.mp4'  # Update to video path if needed
cap = cv2.VideoCapture(video_source)

# Define the output video writer (optional)
frame_width = int(cap.get(3))
frame_height = int(cap.get(4))
out = cv2.VideoWriter('F:/insighteye/datacenter/helmet/trafficTestOutPut3.mp4', cv2.VideoWriter_fourcc(*'XVID'), 30, (frame_width, frame_height))

# Define colors for classes
colors = {
    'person': (0, 255, 0),  # Green for person
    'motorcycle': (255, 0, 0),  # Blue for motorcycle
    'helmet': (0, 0, 255),  # Red for helmet
}

# Class IDs for COCO dataset
person_class_id = 0  # Replace with your model's person class ID
motorcycle_class_id = 3  # Replace with your model's motorcycle class ID
helmet_class_id = 1  # Replace with your model's helmet class ID

# Function to calculate IoU (Intersection over Union)
def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

    union = area1 + area2 - intersection
    return intersection / union if union > 0 else 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Perform inference on the frame
    results = model(frame)

    person_count = 0
    motorcycle_count = 0
    helmet_count = 0
    helmetless_count = 0

    person_boxes = []
    helmet_boxes = []

    for result in results:
        for box in result.boxes.data:
            x1, y1, x2, y2 = map(int, box[:4])  # Bounding box coordinates
            conf = float(box[4])                # Confidence score (as float)
            cls = int(box[5])                   # Class ID (as integer)

            # Detect person
            if cls == person_class_id:
                person_count += 1
                person_boxes.append((x1, y1, x2, y2))
                label = f'Person {conf:.2f}'
                color = colors['person']
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            # Detect motorcycle
            elif cls == motorcycle_class_id:
                motorcycle_count += 1
                label = f'Motorcycle {conf:.2f}'
                color = colors['motorcycle']
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

            # Detect helmet
            elif cls == helmet_class_id:
                helmet_count += 1
                helmet_boxes.append((x1, y1, x2, y2))
                label = f'Helmet {conf:.2f}'
                color = colors['helmet']
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Check if persons are wearing helmets
    for person_box in person_boxes:
        helmet_found = False
        for helmet_box in helmet_boxes:
            iou = calculate_iou(person_box, helmet_box)
            if iou > 0.3:  # Adjust IoU threshold as needed
                helmet_found = True
                break
        if not helmet_found:
            helmetless_count += 1
            x1, y1, x2, y2 = person_box
            cv2.putText(frame, 'No Helmet', (x1, y1 - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)

    # Display the counts on the frame
    cv2.putText(frame, f'Persons Detected: {person_count}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.putText(frame, f'Motorcycles Detected: {motorcycle_count}', (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
    cv2.putText(frame, f'Helmetless Persons: {helmetless_count}', (10, 110), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

    # Show the video frame
    cv2.imshow('Helmet Detection', frame)

    # Write the frame to output video (optional)
    out.write(frame)

    # Break the loop on 'q' key press
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
out.release()
cv2.destroyAllWindows()



0: 384x640 1 Without_Helmet, 814.0ms
Speed: 3.0ms preprocess, 814.0ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 757.2ms
Speed: 5.0ms preprocess, 757.2ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 700.8ms
Speed: 2.0ms preprocess, 700.8ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 724.9ms
Speed: 3.0ms preprocess, 724.9ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 715.7ms
Speed: 2.0ms preprocess, 715.7ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 711.7ms
Speed: 3.0ms preprocess, 711.7ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 737.2ms
Speed: 2.0ms preprocess, 737.2ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 710.8ms
Speed: 3.0ms prep